# Basics of Natural Language Processing (NLP): Worked Take-Home Exercise

> This is a worked learning copy. Review the results, change the sample text, and explain the observations in your own words before treating the exercise as complete.


Use the following link to find open source data sets to complete take-home exercises.

[Data Sets](https://opendatascience.com/20-open-datasets-for-natural-language-processing/)

Or, you can try out Assignment 1 data set for a head start to the work!

# Run this code in the beginning to limit the output size of the cells

In [ ]:
from IPython.display import display, Javascript

def resize_colab_cell():
    """Limit output height when the notebook is running in Google Colab."""
    if "google.colab" in str(get_ipython()):
        display(Javascript(
            "google.colab.output.setIframeHeight(0, true, {maxHeight: 400})"
        ))

resize_colab_cell()

### 1. Input Text

Write a function to collect text data for the analysis via user input - E.g. from a text box



In [ ]:
DEFAULT_TEXT = (
    "Natural language processing helps computers analyse human language. "
    "It supports applications such as search, translation, summarisation, "
    "and sentiment analysis, but meaning still depends on context."
)

def collect_text(prompt="Enter text for NLP analysis: ", default=DEFAULT_TEXT):
    """Collect non-empty text, using a reproducible default when input is blank."""
    user_text = input(prompt).strip()
    return user_text or default

# Set USE_INTERACTIVE_INPUT to True when you want to type into the text box.
USE_INTERACTIVE_INPUT = False
text = collect_text() if USE_INTERACTIVE_INPUT else DEFAULT_TEXT
print(text)

### 2. Basic Analysis

Perform basic text analysis on the collected text using Spacy ([spacy.io](http://spacy.io)) library.

In [ ]:
import spacy

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    # A blank English pipeline still tokenises; install en_core_web_sm for POS/NER.
    nlp = spacy.blank("en")
    nlp.add_pipe("sentencizer")

doc = nlp(text)
print(f"Characters: {len(text)}")
print(f"Sentences: {sum(1 for _ in doc.sents)}")
print(f"Tokens: {len(doc)}")
print(f"Alphabetic tokens: {sum(token.is_alpha for token in doc)}")

analysis = [
    {
        "token": token.text,
        "lemma": token.lemma_ or token.text.lower(),
        "pos": token.pos_ or "model not installed",
        "is_stop": token.is_stop,
    }
    for token in doc
]
analysis[:12]

### 3. Tokenizer
Create a custom tokenizer in Python that handles:
*   Contractions (e.g., "don't" → "do n't")
*   Keeps punctuation as separate tokens
*   Splits hyphenated words (e.g., "state-of-the-art" → "state of the art")

Compare its results with NLTK's word_tokenize on any sample paragraph and the following examples:
"New York-based company", "It's a beautiful day!", "https://www.example.com"

What differences do you see? What are the advantages, and limitations of each approach?

In [ ]:
import re
import nltk
from nltk.tokenize import word_tokenize

for resource in ["punkt", "punkt_tab"]:
    nltk.download(resource, quiet=True)

# URLs are matched first so their punctuation is not split. Apostrophe suffixes,
# words/numbers, and remaining punctuation are then captured separately.
TOKEN_PATTERN = re.compile(
    r"https?://[^\s]+|www\.[^\s]+|n't|'(?:s|re|ve|ll|d|m)|[A-Za-z]+|\d+(?:\.\d+)?|[^\w\s]",
    flags=re.IGNORECASE,
)

def custom_tokenize(value):
    # Separate n't before token matching; e.g. don't -> do + n't.
    value = re.sub(r"\b([A-Za-z]+)n['’]t\b", r"\1 n't", value, flags=re.IGNORECASE)
    # Treat hyphens as boundaries, as required by the exercise.
    value = re.sub(r"(?<=\w)-(?=\w)", " ", value)
    return TOKEN_PATTERN.findall(value)

examples = [
    "New York-based company",
    "It's a beautiful day!",
    "I don't think the state-of-the-art model is ready.",
    "https://www.example.com",
]

for example in examples:
    print(f"Text:   {example}")
    print(f"Custom: {custom_tokenize(example)}")
    print(f"NLTK:   {word_tokenize(example)}\n")

print(
    "Observation: the custom rules deliberately split hyphenated compounds and "
    "preserve complete URLs. NLTK handles common English contractions well, but "
    "its general rules do not exactly match this task. The custom tokenizer is "
    "transparent and task-specific, although it needs more rules for Unicode, "
    "email addresses, unusual apostrophes, and other languages."
)

### 4. Regex

Try writing your own RegEx that can capture citations in text E.g. (Horning, 2022)

In [ ]:
import re

citation_text = (
    "Tokenisation decisions affect later analysis (Horning, 2022). "
    "This has also been discussed in earlier work (Bird et al., 2009; Jurafsky & Martin, 2025)."
)

# Capture one or more author-year references inside parentheses.
citation_pattern = re.compile(
    r"\((?:[A-Z][A-Za-z'’-]+(?:\s+(?:et al\.|&\s+[A-Z][A-Za-z'’-]+))?,\s*\d{4}[a-z]?"
    r"(?:;\s*)?)+\)"
)

citations = citation_pattern.findall(citation_text)
citations

Extract URLS following a certain format (www. or http or https:// ..)

In [ ]:
url_text = (
    "Course information is at https://www.uts.edu.au/courses/master-of-data-science-and-innovation, "
    "documentation is at http://spacy.io, and an example is www.example.com/test?q=nlp."
)

url_pattern = re.compile(r"(?:https?://|www\.)[^\s<>()]+", re.IGNORECASE)
urls = [match.rstrip(".,;:!?\")'") for match in url_pattern.findall(url_text)]
urls

### 5. Word Frequency

Find the list of words that occur more than 10 times in a selected corpus.

Try using different forms of setup: no stopwords, custom stopwords, not removing punctuation, etc. and see what difference in results they produce.


In [ ]:
from collections import Counter
import string
import nltk
from nltk.corpus import gutenberg, stopwords
from nltk.tokenize import word_tokenize

for resource in ["gutenberg", "stopwords", "punkt_tab"]:
    nltk.download(resource, quiet=True)

# Jane Austen's Emma is a reproducible public-domain corpus bundled through NLTK.
corpus_text = gutenberg.raw("austen-emma.txt")
tokens = [token.lower() for token in word_tokenize(corpus_text)]
english_stopwords = set(stopwords.words("english"))
custom_stopwords = english_stopwords | {"said", "mr", "mrs", "emma"}

def frequent_words(items, excluded=None, remove_punctuation=True, threshold=10):
    excluded = excluded or set()
    filtered = [
        token for token in items
        if token not in excluded
        and (not remove_punctuation or token.isalpha())
    ]
    return [item for item in Counter(filtered).most_common() if item[1] > threshold]

setups = {
    "no stopword removal": frequent_words(tokens),
    "standard stopwords removed": frequent_words(tokens, english_stopwords),
    "custom stopwords removed": frequent_words(tokens, custom_stopwords),
    "punctuation retained": frequent_words(tokens, english_stopwords, False),
}

for name, counts in setups.items():
    print(f"{name}: {len(counts)} terms occur more than 10 times")
    print(counts[:15], "\n")

print(
    "Observation: without stopword removal, grammatical function words dominate. "
    "Standard stopwords reveal content words; custom stopwords suppress corpus-specific "
    "high-frequency names. Retaining punctuation makes punctuation marks some of the "
    "most frequent tokens, which is rarely useful for a topical frequency analysis."
)

### 6. Web scraping

Extract the core subjects from this web page: https://www.uts.edu.au/courses/master-of-data-science-and-innovation

In [ ]:
import re
import requests
from bs4 import BeautifulSoup

course_url = "https://www.uts.edu.au/courses/master-of-data-science-and-innovation"
response = requests.get(
    course_url,
    headers={"User-Agent": "Mozilla/5.0 (educational NLP exercise)"},
    timeout=30,
)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")

# Locate four-digit subject codes and names in table rows. This avoids relying
# on fragile CSS class names generated by the site's presentation framework.
core_subjects = []
for row in soup.find_all("tr"):
    cells = [cell.get_text(" ", strip=True) for cell in row.find_all(["th", "td"])]
    if len(cells) >= 2 and re.fullmatch(r"36\d{3}", cells[0]):
        core_subjects.append({"code": cells[0], "subject": cells[1]})

# Fallback for a page redesign that renders table content without <tr> elements.
if not core_subjects:
    page_text = soup.get_text(" ", strip=True)
    matches = re.findall(r"(36\d{3})\s+([A-Z][A-Za-z &-]+?)(?=\s+36\d{3}|$)", page_text)
    core_subjects = [{"code": code, "subject": name.strip()} for code, name in matches]

core_subjects